# §4 Individual with ML#2 filter (v12 top-50 raw-Sharpe, V4 combo-agnostic filter)

Per-combo metrics and equity/drawdown curves after applying the
ML#2 booster + pooled-R:R isotonic calibrator filter.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=0.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_raw_sharpe_top50.json', version='v4_no_gcid')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_raw_sharpe_top50.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Loaded 50 strategies.
Loaded results_raw from cache (50 combos).
Combined unfiltered trades: 43,846
Loaded combos_ml2 from cache (50 combos).
ML2 portfolio trade counts: {'fixed_dollars_500': 1645}


In [2]:
rows = []
for cid, entry in s4_pnl_by_combo.items():
    pnl_base = entry['pnl_base']; risk_base = entry['risk_base']
    if len(pnl_base) == 0:
        for policy in POLICIES:
            rows.append({'combo_id': cid, 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = entry['by_policy'][policy]
        rows.append({'combo_id': cid, 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf4 = pd.DataFrame(rows)
perf4

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_7872,fixed_dollars_500,12,8.2,0.0833,-881.72,-1.76,-0.3697,4.23,2169.21
1,v11_23634,fixed_dollars_500,53,36.3,0.0755,-5703.90,-11.41,-1.2427,11.41,5703.90
2,v11_15760,fixed_dollars_500,1,0.7,0.0000,-484.03,-0.97,0.0000,0.97,484.03
3,v11_2646,fixed_dollars_500,17,11.6,0.1765,-49.13,-0.10,-0.0184,2.70,1349.01
4,v11_7877,fixed_dollars_500,22,15.1,0.2273,-893.83,-1.79,-0.2522,4.90,2473.88
5,v11_11149,fixed_dollars_500,12,8.2,0.3333,-1278.01,-2.56,-0.6950,4.73,2416.89
6,v11_14258,fixed_dollars_500,19,13.0,0.2105,163.49,0.33,0.0322,6.23,3334.82
7,v11_694,fixed_dollars_500,102,69.8,0.5000,793.04,1.59,0.1624,7.85,4167.54
8,v11_9978,fixed_dollars_500,47,32.2,0.2128,1849.08,3.70,0.3980,5.05,2522.64
9,v11_8640,fixed_dollars_500,15,10.3,0.2667,2543.81,5.09,0.5652,3.74,1952.52


In [3]:
plot_ml2_indiv_equity(s4_pnl_by_combo, bars, 'fixed_dollars_500')

In [4]:
plot_ml2_indiv_dd(s4_pnl_by_combo, bars, 'fixed_dollars_500')